<a href="https://colab.research.google.com/github/kaparow/AutoMapForPipeLine/blob/main/Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from scipy.interpolate import RegularGridInterpolator
import plotly.graph_objects as go


# ============================================================
# 1. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================

def cumulative_distance(x, y):
    """Накопленное расстояние вдоль трассы."""
    ds = np.sqrt(np.diff(x) ** 2 + np.diff(y) ** 2)
    return np.concatenate([[0], np.cumsum(ds)])


def format_picket(distance_m):
    """
    Перевод расстояния вдоль трассы в пикетаж.

    Примеры:
    0 м     -> ПК0+00
    85 м    -> ПК0+85
    185 м   -> ПК1+85
    1068.2 м -> ПК10+68.2
    """
    total = round(float(distance_m), 1)

    pk = int(total // 100)
    plus = total - pk * 100

    if plus >= 99.95:
        pk += 1
        plus = 0.0

    if abs(plus - round(plus)) < 1e-9:
        return f"ПК{pk}+{int(round(plus)):02d}"

    return f"ПК{pk}+{plus:04.1f}"


def indices_by_station(s_m, step_m):
    """
    Возвращает индексы точек трассы через заданный шаг по расстоянию.
    Например, через каждые 100 м для пикетов или через 35 м для опор.
    """
    stations = np.arange(0, s_m[-1] + step_m, step_m)
    indices = [int(np.argmin(np.abs(s_m - s))) for s in stations]
    return sorted(set(indices))


def interpolate_by_distance(s_target, s_m, x, y, z):
    """
    Интерполяция координат X, Y, Z по расстоянию вдоль трассы.
    Используется для точного размещения пикетных отметок.
    """
    x_i = np.interp(s_target, s_m, x)
    y_i = np.interp(s_target, s_m, y)
    z_i = np.interp(s_target, s_m, z)
    return x_i, y_i, z_i


def arc(x0, y0, heading_from, heading_to, r=20, n=35):
    """
    Плавная дуга поворота из точки (x0, y0).
    Углы задаются в радианах.
    """
    delta = (heading_to - heading_from + np.pi) % (2 * np.pi) - np.pi
    perp = heading_from + np.pi / 2 if delta > 0 else heading_from - np.pi / 2

    cx = x0 + r * np.cos(perp)
    cy = y0 + r * np.sin(perp)

    a_start = np.arctan2(y0 - cy, x0 - cx)
    angles = np.linspace(a_start, a_start + delta, n)

    x = cx + r * np.cos(angles)
    y = cy + r * np.sin(angles)

    return x, y


# ============================================================
# 2. ТРАССА ТРУБОПРОВОДА
# ============================================================

def build_pipeline_route():
    """
    Формирует удлинённую трассу:
    восток -> север -> восток -> юг -> восток
    """
    R = 22
    segments_x, segments_y = [], []

    E = 0.0
    N = np.pi / 2
    S = -np.pi / 2

    # 1) Восток 220 м
    t = np.linspace(0, 220, 130)
    segments_x.append(t)
    segments_y.append(np.zeros_like(t))

    # Поворот 1: восток -> север
    ax, ay = arc(220, 0, E, N, r=R, n=40)
    segments_x.append(ax)
    segments_y.append(ay)

    # 2) Север 150 м
    x0, y0 = ax[-1], ay[-1]
    t = np.linspace(0, 150, 95)
    segments_x.append(np.full_like(t, x0))
    segments_y.append(y0 + t)

    # Поворот 2: север -> восток
    x0, y0 = segments_x[-1][-1], segments_y[-1][-1]
    ax, ay = arc(x0, y0, N, E, r=R, n=40)
    segments_x.append(ax)
    segments_y.append(ay)

    # 3) Восток 260 м
    x0, y0 = ax[-1], ay[-1]
    t = np.linspace(0, 260, 155)
    segments_x.append(x0 + t)
    segments_y.append(np.full_like(t, y0))

    # Поворот 3: восток -> юг
    x0, y0 = segments_x[-1][-1], segments_y[-1][-1]
    ax, ay = arc(x0, y0, E, S, r=R, n=40)
    segments_x.append(ax)
    segments_y.append(ay)

    # 4) Юг 120 м
    x0, y0 = ax[-1], ay[-1]
    t = np.linspace(0, 120, 80)
    segments_x.append(np.full_like(t, x0))
    segments_y.append(y0 - t)

    # Поворот 4: юг -> восток
    x0, y0 = segments_x[-1][-1], segments_y[-1][-1]
    ax, ay = arc(x0, y0, S, E, r=R, n=40)
    segments_x.append(ax)
    segments_y.append(ay)

    # 5) Восток 180 м
    x0, y0 = ax[-1], ay[-1]
    t = np.linspace(0, 180, 110)
    segments_x.append(x0 + t)
    segments_y.append(np.full_like(t, y0))

    x_pipe = np.concatenate(segments_x)
    y_pipe = np.concatenate(segments_y)

    return x_pipe, y_pipe


# ============================================================
# 3. УСЛОВНЫЙ РЕЛЬЕФ
# ============================================================

def build_terrain(x_pipe, y_pipe):
    """
    Полностью ровная условная поверхность земли.
    Используется как демонстрационный фон для трассировочной карты.
    """
    pad = 55

    xg = np.linspace(x_pipe.min() - pad, x_pipe.max() + pad, 180)
    yg = np.linspace(y_pipe.min() - pad, y_pipe.max() + pad, 160)
    Xg, Yg = np.meshgrid(xg, yg)

    Zg = np.zeros_like(Xg)

    return xg, yg, Zg


# ============================================================
# 4. ВЫСОТНЫЙ ПРОФИЛЬ ТРУБОПРОВОДА
# ============================================================

def apply_elevated_section(clearance, s_m, s_start, up_len, top_len, down_len, top_level):
    """
    Добавляет поднятый участок:
    подъём -> верхняя полка -> спуск.
    Все параметры задаются вдоль расстояния s_m.
    """
    base_local = np.interp(s_start, s_m, clearance)

    s_up_end = s_start + up_len
    s_top_end = s_up_end + top_len
    s_down_end = s_top_end + down_len

    mask_up = (s_m >= s_start) & (s_m < s_up_end)
    mask_top = (s_m >= s_up_end) & (s_m <= s_top_end)
    mask_down = (s_m > s_top_end) & (s_m <= s_down_end)

    if mask_up.any():
        clearance[mask_up] = np.maximum(
            clearance[mask_up],
            np.linspace(base_local, top_level, mask_up.sum())
        )

    if mask_top.any():
        clearance[mask_top] = np.maximum(clearance[mask_top], top_level)

    if mask_down.any():
        target = np.linspace(top_level, base_local + 0.15, mask_down.sum())
        clearance[mask_down] = np.maximum(clearance[mask_down], target)

    return clearance


def apply_small_hump(clearance, s_m, s_center, half_width, peak_level):
    """
    Добавляет небольшое локальное поднятие.
    """
    mask = (s_m >= s_center - half_width) & (s_m <= s_center + half_width)

    if not mask.any():
        return clearance

    idx = np.where(mask)[0]
    t = np.linspace(0, 1, len(idx))

    hump = 2.9 + (peak_level - 2.9) * np.sin(np.pi * t)
    clearance[idx] = np.maximum(clearance[idx], hump)

    return clearance


def build_pipe_height(x_pipe, y_pipe, xg, yg, Zg):
    interp = RegularGridInterpolator(
        (yg, xg),
        Zg,
        bounds_error=False,
        fill_value=0.0
    )

    z_ground = interp(np.column_stack([y_pipe, x_pipe]))

    s_m = cumulative_distance(x_pipe, y_pipe)
    s_norm = s_m / s_m[-1]

    # Базовый проектный просвет трубы над поверхностью земли
    clearance = (
        2.8
        + 0.18 * np.sin(2 * np.pi * s_norm)
        + 0.10 * np.sin(8 * np.pi * s_norm)
    )

    # Основной высокий участок обхода препятствия
    clearance = apply_elevated_section(
        clearance=clearance,
        s_m=s_m,
        s_start=560,
        up_len=22,
        top_len=42,
        down_len=22,
        top_level=7.2
    )

    # Низкие локальные препятствия / изменения профиля
    clearance = apply_small_hump(
        clearance,
        s_m,
        s_center=170,
        half_width=10,
        peak_level=4.6
    )

    clearance = apply_small_hump(
        clearance,
        s_m,
        s_center=360,
        half_width=12,
        peak_level=4.8
    )

    clearance = apply_small_hump(
        clearance,
        s_m,
        s_center=820,
        half_width=12,
        peak_level=4.7
    )

    z_pipe = z_ground + clearance

    return z_pipe, z_ground, s_m


# ============================================================
# 5. ОБНАРУЖЕНИЕ ХАРАКТЕРНЫХ ТОЧЕК
# ============================================================

def detect_turns_by_curvature(x, y, curvature_threshold=0.015, min_group_len=6):
    """
    Определение поворотов по изменению направления трассы.
    """
    dx = np.gradient(x)
    dy = np.gradient(y)

    heading = np.unwrap(np.arctan2(dy, dx))
    curvature = np.abs(np.gradient(heading))

    candidates = np.where(curvature > curvature_threshold)[0]

    if len(candidates) == 0:
        return []

    groups = []
    current_group = [candidates[0]]

    for i in candidates[1:]:
        if i == current_group[-1] + 1:
            current_group.append(i)
        else:
            if len(current_group) >= min_group_len:
                groups.append(current_group)
            current_group = [i]

    if len(current_group) >= min_group_len:
        groups.append(current_group)

    turn_indices = [int(group[len(group) // 2]) for group in groups]

    return turn_indices


def detect_obstacle_segments(z_pipe, z_ground, s_m, clearance_threshold=4.3):
    """
    Определяет участки, где просвет трубы над грунтом превышает порог.
    Возвращает не просто одну точку, а начало, конец и пик препятствия.
    """
    clearance = z_pipe - z_ground
    mask = clearance > clearance_threshold

    obstacles = []
    in_obstacle = False
    start = 0

    for i, is_obstacle in enumerate(mask):
        if is_obstacle and not in_obstacle:
            start = i
            in_obstacle = True

        elif not is_obstacle and in_obstacle:
            end = i - 1
            peak = start + int(np.argmax(clearance[start:end + 1]))

            obstacles.append({
                "start_idx": start,
                "end_idx": end,
                "peak_idx": peak,
                "s_start": s_m[start],
                "s_end": s_m[end],
                "s_peak": s_m[peak]
            })

            in_obstacle = False

    if in_obstacle:
        end = len(mask) - 1
        peak = start + int(np.argmax(clearance[start:end + 1]))

        obstacles.append({
            "start_idx": start,
            "end_idx": end,
            "peak_idx": peak,
            "s_start": s_m[start],
            "s_end": s_m[end],
            "s_peak": s_m[peak]
        })

    return obstacles


# ============================================================
# 6. ОПОРЫ И ПИКЕТЫ
# ============================================================

def place_supports_by_distance(s_m, support_step_m=35):
    """
    Размещает условные опоры через заданное расстояние вдоль трассы.
    Это правильнее, чем ставить опоры через количество точек массива.
    """
    return indices_by_station(s_m, support_step_m)


def build_picket_points(s_m, x_pipe, y_pipe, z_ground, step_m=100):
    """
    Формирует точки пикетажа через каждые 100 м.
    """
    picket_distances = np.arange(0, s_m[-1] + 0.01, step_m)

    picket_x = []
    picket_y = []
    picket_z = []
    picket_labels = []
    picket_hover = []

    for s in picket_distances:
        x_i, y_i, z_i = interpolate_by_distance(
            s_target=s,
            s_m=s_m,
            x=x_pipe,
            y=y_pipe,
            z=z_ground
        )

        picket_x.append(x_i)
        picket_y.append(y_i)
        picket_z.append(z_i + 0.15)

        label = f"ПК{int(s // 100)}"
        picket_labels.append(label)

        picket_hover.append(
            f"Пикет: {format_picket(s)}<br>"
            f"Расстояние от начала: {s:.1f} м<br>"
            f"X: {x_i:.1f} м<br>"
            f"Y: {y_i:.1f} м"
        )

    return picket_x, picket_y, picket_z, picket_labels, picket_hover


# ============================================================
# 7. СБОРКА ДАННЫХ
# ============================================================

x_pipe, y_pipe = build_pipeline_route()

xg, yg, Zg = build_terrain(x_pipe, y_pipe)

z_pipe, z_ground_under_pipe, s_m = build_pipe_height(
    x_pipe,
    y_pipe,
    xg,
    yg,
    Zg
)

turn_indices = detect_turns_by_curvature(
    x_pipe,
    y_pipe,
    curvature_threshold=0.015,
    min_group_len=6
)

obstacle_segments = detect_obstacle_segments(
    z_pipe,
    z_ground_under_pipe,
    s_m,
    clearance_threshold=4.3
)

obstacle_indices = [item["peak_idx"] for item in obstacle_segments]

support_indices = place_supports_by_distance(
    s_m,
    support_step_m=35
)

picket_x, picket_y, picket_z, picket_labels, picket_hover = build_picket_points(
    s_m=s_m,
    x_pipe=x_pipe,
    y_pipe=y_pipe,
    z_ground=z_ground_under_pipe,
    step_m=100
)

n_total = len(x_pipe)

char_points = {
    "Начало трассы": 0,
    "Конец трассы": n_total - 1
}

for i, idx in enumerate(turn_indices):
    char_points[f"Поворот {i + 1}"] = idx

for i, idx in enumerate(obstacle_indices):
    char_points[f"Препятствие {i + 1}"] = idx


# ============================================================
# 8. ТЕКСТОВЫЙ ВЫВОД
# ============================================================

print(f"Точек трассы:            {n_total}")
print(f"Длина трассы:            {s_m[-1]:.1f} м ({format_picket(s_m[-1])})")
print(f"Опор:                    {len(support_indices)}")
print(f"Поворотов:               {len(turn_indices)}")
print(f"Препятствий:             {len(obstacle_indices)}")
print(f"Диапазон X:              {x_pipe.min():.1f} -> {x_pipe.max():.1f} м")
print(f"Диапазон Y:              {y_pipe.min():.1f} -> {y_pipe.max():.1f} м")
print(f"Мин. отметка грунта:     {z_ground_under_pipe.min():.2f} м")
print(f"Макс. отметка грунта:    {z_ground_under_pipe.max():.2f} м")
print(f"Мин. просвет:            {(z_pipe - z_ground_under_pipe).min():.2f} м")
print(f"Макс. просвет:           {(z_pipe - z_ground_under_pipe).max():.2f} м")

print("\nХарактерные точки трассы:")
for label, idx in char_points.items():
    print(
        f"{label:20s} | "
        f"{format_picket(s_m[idx]):10s} | "
        f"s = {s_m[idx]:7.1f} м | "
        f"X = {x_pipe[idx]:7.1f} м | "
        f"Y = {y_pipe[idx]:7.1f} м | "
        f"Z = {z_pipe[idx]:5.2f} м"
    )

print("\nПрепятствия по участкам:")
for i, item in enumerate(obstacle_segments, start=1):
    print(
        f"Препятствие {i}: "
        f"{format_picket(item['s_start'])} — {format_picket(item['s_end'])}, "
        f"пик: {format_picket(item['s_peak'])}"
    )


# ============================================================
# 9. ВИЗУАЛИЗАЦИЯ PLOTLY
# ============================================================

fig = go.Figure()

# Поверхность земли
fig.add_trace(go.Surface(
    x=xg,
    y=yg,
    z=Zg,
    colorscale="YlGn",
    opacity=0.84,
    showscale=False,
    name="Условная поверхность земли"
))

# Проекция оси на грунт
projection_hover = [
    f"Проекция оси<br>"
    f"Пикет: {format_picket(s)}<br>"
    f"Расстояние: {s:.1f} м<br>"
    f"X: {x:.1f} м<br>"
    f"Y: {y:.1f} м"
    for x, y, s in zip(x_pipe, y_pipe, s_m)
]

fig.add_trace(go.Scatter3d(
    x=x_pipe,
    y=y_pipe,
    z=z_ground_under_pipe + 0.03,
    mode="lines",
    line=dict(color="rgba(180, 50, 20, 0.55)", width=4),
    text=projection_hover,
    hovertemplate="%{text}<extra></extra>",
    name="Проекция оси на землю"
))

# Ось трубопровода
pipe_hover = [
    f"Ось трубопровода<br>"
    f"Пикет: {format_picket(s)}<br>"
    f"Расстояние: {s:.1f} м<br>"
    f"X: {x:.1f} м<br>"
    f"Y: {y:.1f} м<br>"
    f"Z: {z:.2f} м<br>"
    f"Просвет: {(z - zg):.2f} м"
    for x, y, z, zg, s in zip(
        x_pipe,
        y_pipe,
        z_pipe,
        z_ground_under_pipe,
        s_m
    )
]

fig.add_trace(go.Scatter3d(
    x=x_pipe,
    y=y_pipe,
    z=z_pipe,
    mode="lines",
    line=dict(color="red", width=8),
    text=pipe_hover,
    hovertemplate="%{text}<extra></extra>",
    name="Ось трубопровода"
))

# Опоры
x_sup_lines, y_sup_lines, z_sup_lines = [], [], []

for idx in support_indices:
    x_sup_lines += [x_pipe[idx], x_pipe[idx], None]
    y_sup_lines += [y_pipe[idx], y_pipe[idx], None]
    z_sup_lines += [z_ground_under_pipe[idx], z_pipe[idx], None]

fig.add_trace(go.Scatter3d(
    x=x_sup_lines,
    y=y_sup_lines,
    z=z_sup_lines,
    mode="lines",
    line=dict(color="saddlebrown", width=5),
    name="Опоры"
))

# Верхние точки опор с hover-подсказками
support_hover = [
    f"Опора<br>"
    f"Пикет: {format_picket(s_m[idx])}<br>"
    f"Расстояние: {s_m[idx]:.1f} м<br>"
    f"Высота трубы: {z_pipe[idx]:.2f} м<br>"
    f"Просвет: {(z_pipe[idx] - z_ground_under_pipe[idx]):.2f} м"
    for idx in support_indices
]

fig.add_trace(go.Scatter3d(
    x=[x_pipe[idx] for idx in support_indices],
    y=[y_pipe[idx] for idx in support_indices],
    z=[z_pipe[idx] for idx in support_indices],
    mode="markers",
    marker=dict(size=4, color="saddlebrown"),
    text=support_hover,
    hovertemplate="%{text}<extra></extra>",
    name="Точки опор"
))

# Пикетные отметки через каждые 100 м
fig.add_trace(go.Scatter3d(
    x=picket_x,
    y=picket_y,
    z=picket_z,
    mode="markers+text",
    marker=dict(size=5, color="black", symbol="circle"),
    text=picket_labels,
    textposition="bottom center",
    hovertext=picket_hover,
    hovertemplate="%{hovertext}<extra></extra>",
    name="Пикеты через 100 м"
))

# Подписи характерных точек
start_end_labels = []
start_end_x = []
start_end_y = []
start_end_z = []

special_labels = []
special_x = []
special_y = []
special_z = []
special_hover = []

for label, idx in char_points.items():
    label_with_pk = f"{label}<br>{format_picket(s_m[idx])}"

    hover_text = (
        f"{label}<br>"
        f"Пикет: {format_picket(s_m[idx])}<br>"
        f"Расстояние: {s_m[idx]:.1f} м<br>"
        f"X: {x_pipe[idx]:.1f} м<br>"
        f"Y: {y_pipe[idx]:.1f} м<br>"
        f"Z: {z_pipe[idx]:.2f} м<br>"
        f"Просвет: {(z_pipe[idx] - z_ground_under_pipe[idx]):.2f} м"
    )

    if "Поворот" in label or "Препятствие" in label:
        special_labels.append(label_with_pk)
        special_x.append(x_pipe[idx])
        special_y.append(y_pipe[idx])
        special_z.append(z_pipe[idx] + 0.9)
        special_hover.append(hover_text)
    else:
        start_end_labels.append(label_with_pk)
        start_end_x.append(x_pipe[idx])
        start_end_y.append(y_pipe[idx])
        start_end_z.append(z_pipe[idx] + 0.9)

fig.add_trace(go.Scatter3d(
    x=start_end_x,
    y=start_end_y,
    z=start_end_z,
    mode="markers+text",
    marker=dict(size=7, color="royalblue", symbol="circle"),
    text=start_end_labels,
    textposition="top center",
    name="Начало / конец"
))

fig.add_trace(go.Scatter3d(
    x=special_x,
    y=special_y,
    z=special_z,
    mode="markers+text",
    marker=dict(size=6, color="darkorange", symbol="diamond"),
    text=special_labels,
    textposition="top center",
    hovertext=special_hover,
    hovertemplate="%{hovertext}<extra></extra>",
    name="Повороты / препятствия"
))

fig.update_layout(
    title="Рисунок 1.3 — Трассировочная карта надземного трубопровода с пикетажем",
    scene=dict(
        xaxis_title="X, м",
        yaxis_title="Y, м",
        zaxis_title="Высота, м",
        aspectratio=dict(x=3.2, y=1.7, z=0.9),
        camera=dict(eye=dict(x=1.8, y=-1.9, z=1.05))
    ),
    legend=dict(
        x=0.01,
        y=0.98,
        bgcolor="rgba(255,255,255,0.86)"
    ),
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()

Точек трассы:            730
Длина трассы:            1068.2 м (ПК10+68.2)
Опор:                    32
Поворотов:               4
Препятствий:             4
Диапазон X:              0.0 -> 748.0 м
Диапазон Y:              0.0 -> 194.0 м
Мин. отметка грунта:     0.00 м
Макс. отметка грунта:    0.00 м
Мин. просвет:            2.53 м
Макс. просвет:           7.20 м

Характерные точки трассы:
Начало трассы        | ПК0+00     | s =     0.0 м | X =     0.0 м | Y =     0.0 м | Z =  2.80 м
Конец трассы         | ПК10+68.2  | s =  1068.2 м | X =   748.0 м | Y =    30.0 м | Z =  2.80 м
Поворот 1            | ПК2+37.7   | s =   237.7 м | X =   235.9 м | Y =     6.8 м | Z =  2.91 м
Поворот 2            | ПК4+22.3   | s =   422.3 м | X =   248.8 м | Y =   187.9 м | Z =  2.86 м
Поворот 3            | ПК7+16.8   | s =   716.8 м | X =   539.9 м | Y =   187.2 м | Z =  2.55 м
Поворот 4            | ПК8+71.4   | s =   871.4 м | X =   552.8 м | Y =    36.1 м | Z =  2.73 м
Препятствие 1        | ПК1+68.8 